# 100 Annotated TOS LLM Evaluation
TODO: implement a really dumb and stupid evaluation using cleaned_tos_comments.csv. just use f1 score for now. 

## Setup

Import dependencies

In [3]:
import pandas as pd
import numpy as np

In [2]:
DATASET_PATH = '../../datasets/Annotated Terms of Service of 100 Online Platforms/'
GENERATED_FILES_PATH = '../../generated_files/100_tos/'

Prepare the variables and data to give to LLM for reference

1. Variable classifications:

In [4]:
variable_classifications = {
    # Evaluative variables
    "ltd": "Limitation of company's liability",
    "ltd_cap": "Maximal threshold of company's liability", 
    "period": "Limitation period",
    "as_is": "No promises",
    "indemn": "Indemnification clause",
    "c_law": "Choice of law other than user's domicile",
    "c_forum": "Choice of forum other than user's domicile",
    "arb": "Mandatory arbitration",
    "class": "Class action waiver",
    "contr_chg": "Unilateral change of contract",
    "price_chg": "Unilateral change of future prices",
    "serv_chg": "Unilateral change of service by the company",
    "acc_del": "Account deletion and unilateral termination of contract by the company",
    "transfer": "Transfer of contractual rights to another subject",
    "cnt_del": "User content deletion",
    "acc_sus": "Account suspension",
    "recom": "Main parameters used in recommender systems",
    "com_sys": "Internal complaint-handling system",
    "cnt_retr": "Retrieval of digital content by the user",
    "IP": "Excessive user content IP license",
    "discret": "Discretional power to interpret the ToS",
    "interpret": "General interpretation clause",
    "sever": "Severability clauses",
    "suggest": "Right to incorporate user's feedback or suggestions without compensation",
    
    # Count variables
    "uncle": "Unclear law clause",
    "docu": "Other documents",
    
    # Pull-out text variables
    "core": "Promises and obligations",
    "what": "Description of the service",
    
    # Metadata
    "name": "Name of the service",
    "url": "ToS URL",
    "date": "Date effective",
    "lang": "ToS Language",
    "sector": "Service sector",
    "word_cnt": "Number of words",
    "hq": "Company's headquarter",
    "hq_cat": "Headquarter category",
    "public": "Publicly listed",
    "paid": "Paid or free services"
}

Grab the scoring system from `Variables Definitions.xlsx` and the meanings for each score.

In [9]:
evaluative_var_df = pd.read_excel(DATASET_PATH + 'Variables Definitions.xlsx', sheet_name="Evaluative Variables")
evaluative_var_df.head()

,General category,Variable name,Legal ground,Code,Score,Detailed description
0,Limitation of remedy clauses,Limitation of company's liability,Annex 1(a) Directive 93/13,ltd,-1.0,Unlawful limitation of liablility for damages ...
1,NaN,NaN,NaN,NaN,0.0,Lawful limitation - any other than unlawful li...
2,NaN,NaN,NaN,NaN,1.0,Lack of limitation
3,NaN,Maximal threshold of company's liability,Annex 1(a) Directive 93/13,ltd_cap,-1.0,Everyone entitled to the damages connected wit...
4,NaN,NaN,NaN,NaN,0.0,Only Businesses entitled to the damages connec...


In [11]:
count_var_df = pd.read_excel(DATASET_PATH + 'Variables Definitions.xlsx', sheet_name="Count Variables")
count_var_df.head()

,Variable name,Code,Short description
0,Unclear law clause,uncle,Number of clauses making it hard for the consu...
1,Other documents,docu,Number of legal documents other than ToS that ...


In [12]:
pull_out_text_df = pd.read_excel(DATASET_PATH + 'Variables Definitions.xlsx', sheet_name="Pull-out Text Variables")
pull_out_text_df.head()

,Variable name,Code,Short description
0,Promises and obligations,core,Any contractual duties or commitments of the c...
1,Description of the service,what,"Clauses that describe functionalities, feature..."


In [13]:
metadata_df = pd.read_excel(DATASET_PATH + 'Variables Definitions.xlsx', sheet_name="Metadata")
metadata_df.head()

,Variable name,Code,Short description
0,Name of the service,name,The name of the service of which the Terms of ...
1,ToS URL,url,"Link to a page, from which Terms of Service ha..."
2,Date effective,date,Date from which the Terms of Service applies
3,ToS Language,lang,The language in which the Terms of Service was...
4,Service sector,sector,Market sector of the service which the analyze...


What does model need to determine:
- which variable name (or the code)
- what score to give. 

WHAT if we just do score first. give it the variable name already.

(skip count, pull-out text, and metadata for now?)

## Start LLM evaluation Pipeline here: 

In [6]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

load_dotenv()

OLLAMA_API_KEY = os.environ.get("OLLAMA_API_KEY")

if not OLLAMA_API_KEY:
    print("OLLAMA_API_KEY is still not set. Check .env path/value, then re-run this cell.")

In [9]:
import re
from pathlib import Path

comments_rel = Path("generated_files/100_tos/cleaned_tos_comments.csv")
results_rel = Path("datasets/Annotated Terms of Service of 100 Online Platforms/Terms of Service Analysis and Evaluation_RESULTS.csv")
rubric_rel = Path("datasets/Annotated Terms of Service of 100 Online Platforms/Variables Definitions.xlsx")

candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
repo_root = None
for root in candidate_roots:
    if (root / comments_rel).exists() and (root / results_rel).exists() and (root / rubric_rel).exists():
        repo_root = root
        break

if repo_root is None:
    raise FileNotFoundError("Could not resolve repository root containing required evaluation files.")

comments_path = repo_root / comments_rel
results_path = repo_root / results_rel
rubric_path = repo_root / rubric_rel

comments_df = pd.read_csv(comments_path, encoding="utf-8-sig")
comments_df.columns = [str(c).replace("\ufeff", "").strip() for c in comments_df.columns]

results_df = pd.read_csv(results_path, sep=";", encoding="utf-8-sig")
results_df.columns = [str(c).replace("\ufeff", "").strip() for c in results_df.columns]

rubric_df = pd.read_excel(rubric_path, sheet_name="Evaluative Variables")
rubric_df.columns = [str(c).strip() for c in rubric_df.columns]

rubric_df["Code"] = rubric_df["Code"].ffill()
rubric_df["Variable name"] = rubric_df["Variable name"].ffill()
rubric_df = rubric_df.dropna(subset=["Code", "Score", "Detailed description"]).copy()
rubric_df["Code"] = rubric_df["Code"].astype(str).str.strip().str.lower()
rubric_df["Score"] = pd.to_numeric(rubric_df["Score"], errors="coerce")
rubric_df = rubric_df[rubric_df["Score"].isin([-1, 0, 1])].copy()
rubric_df["Score"] = rubric_df["Score"].astype(int)

evaluative_codes = sorted(rubric_df["Code"].unique(), key=len, reverse=True)
# Use single backslashes: \b is regex word boundary (\\b would match literal "\b")
code_pattern = re.compile(
    r"\b(" + "|".join(re.escape(c) for c in evaluative_codes) + r")\b",
    flags=re.IGNORECASE,
)

def extract_classification_code(comment_text: str):
    text = str(comment_text) if pd.notna(comment_text) else ""
    match = code_pattern.search(text)
    if match:
        return match.group(1).lower()
    return None

comments_df["Classification_Code"] = comments_df["comment"].apply(extract_classification_code)
comments_df = comments_df[comments_df["Classification_Code"].isin(set(evaluative_codes))].copy()
comments_df["Company"] = comments_df["company"].astype(str).str.strip()
comments_df["company_norm"] = comments_df["Company"].str.lower().str.strip()

available_eval_codes = [c for c in evaluative_codes if c in results_df.columns]
gt_long_df = results_df.melt(
    id_vars=["name"],
    value_vars=available_eval_codes,
    var_name="Classification_Code",
    value_name="Ground_Truth_Score",
)

gt_long_df["Classification_Code"] = gt_long_df["Classification_Code"].str.lower().str.strip()
gt_long_df["company_norm"] = gt_long_df["name"].astype(str).str.lower().str.strip()
gt_long_df["Ground_Truth_Score"] = pd.to_numeric(gt_long_df["Ground_Truth_Score"], errors="coerce")
gt_long_df = gt_long_df[gt_long_df["Ground_Truth_Score"].isin([-1, 0, 1])].copy()
gt_long_df["Ground_Truth_Score"] = gt_long_df["Ground_Truth_Score"].astype(int)

rubric_text_map = (
    rubric_df.sort_values(["Code", "Score"])
    .groupby("Code")
    .apply(
        lambda g: "\n".join([f"{int(row['Score'])}: {str(row['Detailed description']).strip()}" for _, row in g.iterrows()]),
        include_groups=False  # <-- This silences the warning and future-proofs the code
    )
    .to_dict()
)

merged_df = comments_df.merge(
    gt_long_df[["company_norm", "Classification_Code", "Ground_Truth_Score"]],
    on=["company_norm", "Classification_Code"],
    how="inner",
)

merged_df["Specific_Rubric"] = merged_df["Classification_Code"].map(rubric_text_map)

unified_eval_df = (
    merged_df[["Company", "Classification_Code", "referenced_text", "Specific_Rubric", "Ground_Truth_Score"]]
    .rename(columns={"referenced_text": "Referenced_Text"})
    .dropna(subset=["Referenced_Text", "Specific_Rubric", "Ground_Truth_Score"])
    .reset_index(drop=True)
)

print(f"Resolved repository root: {repo_root}")
print(f"Rows in cleaned comments: {len(pd.read_csv(comments_path, encoding='utf-8-sig'))}")
print(f"Rows after evaluative-code extraction: {len(comments_df)}")
print(f"Rows in unified evaluation dataframe: {len(unified_eval_df)}")
unified_eval_df.head()

Resolved repository root: /Users/riki/Coding Projects/Thesis/lawgic
Rows in cleaned comments: 2640
Rows after evaluative-code extraction: 1565
Rows in unified evaluation dataframe: 1448


,Company,Classification_Code,Referenced_Text,Specific_Rubric,Ground_Truth_Score
0,YouTube,serv_chg,YouTube is constantly changing and improving t...,-1: When the company reserves the right to cha...,0
1,YouTube,cnt_del,If we reasonably believe that any of your Cont...,-1: When a company reserves a right to delete ...,0
2,YouTube,acc_sus,YouTube reserves the right to suspend or termi...,-1: When the company reserves the right to sus...,0
3,YouTube,acc_del,YouTube reserves the right to suspend or termi...,-1: When the company reserves the right to del...,0
4,YouTube,ltd,"To the extent permitted by applicable law, You...",-1: Unlawful limitation of liablility for dama...,0


In [19]:
from typing import Literal

class OrdinalScoreOutput(BaseModel):
    score: Literal[-1, 0, 1] = Field(
        description="Ordinal severity score. Must be exactly one of -1, 0, or 1."
    )

llm = ChatOllama(
    model="gemma4:31b",
    base_url="https://ollama.com",
    temperature=0.0,
    format="json",
    client_kwargs={
        "headers": {
            "Authorization": f"Bearer {OLLAMA_API_KEY}",
        }
    },
)

structured_llm = llm.with_structured_output(OrdinalScoreOutput)

prompt = PromptTemplate.from_template(
    """
You are a strict evaluator for Terms of Service clauses.
Assign exactly one ordinal score: -1, 0, or 1.
Use only provided specific rubric for this classification.
Do not explain reasoning.

Company: {Company}
Classification_Code: {Classification_Code}
Clause:
{Referenced_Text}

Specific_Rubric:
{Specific_Rubric}

Return strict JSON only in this exact shape:
{{"score": -1}}
or
{{"score": 0}}
or
{{"score": 1}}
""".strip()
)

evaluation_chain = prompt | structured_llm
print("LangChain + Ollama Cloud pipeline ready (model: gemma4:31b, format=json).")

LangChain + Ollama Cloud pipeline ready (model: gemma4:31b, format=json).


In [20]:
from sklearn.metrics import f1_score

def _recover_score_from_parse_error(exc: Exception):
    """Recover -1/0/1 when model returns text like `score: 1` instead of JSON."""
    import re

    text = str(exc)
    if "Invalid json output:" not in text:
        return None

    first_line = text.splitlines()[0]
    raw_output = first_line.split("Invalid json output:", 1)[-1].strip()
    match = re.search(r"(?<!\d)(-1|0|1)(?!\d)", raw_output)
    if not match:
        return None
    return int(match.group(1))


def run_llm_evaluation(num_rows: int | None = None, output_filename: str = "evaluated_tos.csv"):
    """
    Run evaluation on unified_eval_df.

    Args:
        num_rows: Number of rows to evaluate. If None, evaluate all rows.
        output_filename: Output CSV filename in generated_files/100_tos.

    Returns:
        final_eval_df, macro_f1_or_none
    """
    if num_rows is None:
        eval_df = unified_eval_df.copy()
    else:
        if num_rows <= 0:
            raise ValueError("num_rows must be a positive integer or None.")
        eval_df = unified_eval_df.head(num_rows).copy()

    total_rows = len(eval_df)
    print(f"Starting inference for {total_rows} rows...")

    predicted_scores = []
    for idx, row in eval_df.iterrows():
        payload = {
            "Company": row["Company"],
            "Classification_Code": row["Classification_Code"],
            "Referenced_Text": row["Referenced_Text"],
            "Specific_Rubric": row["Specific_Rubric"],
        }

        predicted_score = None
        try:
            response = evaluation_chain.invoke(payload)
            predicted_score = int(response.score)
        except Exception as exc:
            recovered_score = _recover_score_from_parse_error(exc)
            if recovered_score in (-1, 0, 1):
                predicted_score = recovered_score
                print(
                    f"[{idx + 1}/{total_rows}] parser recovery used -> {predicted_score}"
                )
            else:
                print(f"[{idx + 1}/{total_rows}] ERROR ({type(exc).__name__}): {exc}")

        predicted_scores.append(predicted_score)
        print(
            f"[{idx + 1}/{total_rows}] "
            f"{row['Company']} | {row['Classification_Code']} -> {predicted_score}"
        )

    final_eval_df = eval_df.copy()
    final_eval_df["Predicted_Score"] = predicted_scores

    output_path = repo_root / "generated_files/100_tos" / output_filename
    output_path.parent.mkdir(parents=True, exist_ok=True)
    final_eval_df.to_csv(output_path, index=False)
    print(f"Saved evaluated rows to: {output_path}")

    valid_mask = (
        final_eval_df["Ground_Truth_Score"].isin([-1, 0, 1])
        & final_eval_df["Predicted_Score"].isin([-1, 0, 1])
    )

    macro_f1 = None
    print(f"Rows included in report: {len(final_eval_df)}")
    if valid_mask.any():
        macro_f1 = f1_score(
            final_eval_df.loc[valid_mask, "Ground_Truth_Score"].astype(int),
            final_eval_df.loc[valid_mask, "Predicted_Score"].astype(int),
            average="macro",
        )
        print(f"Macro F1 (valid predictions only): {macro_f1:.4f}")
        print(f"Valid rows used for F1: {int(valid_mask.sum())}/{len(final_eval_df)}")
    else:
        print("No valid predictions available yet for F1 calculation.")

    return final_eval_df, macro_f1

Test evaluation run

In [ ]:
# Quick test run (example): evaluate first 25 rows
quick_eval_df, quick_macro_f1 = run_llm_evaluation(num_rows=10, output_filename="evaluated_tos_quick.csv")
quick_eval_df

Starting inference for 10 rows...
[1/10] YouTube | serv_chg -> 0
[2/10] YouTube | cnt_del -> 0
[3/10] YouTube | acc_sus -> 0
[4/10] YouTube | acc_del -> 0
[5/10] YouTube | ltd -> 0
[6/10] YouTube | ltd_cap -> -1
[7/10] YouTube | contr_chg -> 1
[8/10] YouTube | sever -> 0
[9/10] SQUARE ENIX | serv_chg -> -1
[10/10] SQUARE ENIX | acc_sus -> -1
Saved evaluated rows to: /Users/riki/Coding Projects/Thesis/lawgic/generated_files/100_tos/evaluated_tos_quick.csv
Rows included in report: 10
Macro F1 (valid predictions only): 0.5524
Valid rows used for F1: 10/10


,Company,Classification_Code,Referenced_Text,Specific_Rubric,Ground_Truth_Score,Predicted_Score
0,YouTube,serv_chg,YouTube is constantly changing and improving t...,-1: When the company reserves the right to cha...,0,0
1,YouTube,cnt_del,If we reasonably believe that any of your Cont...,-1: When a company reserves a right to delete ...,0,0
2,YouTube,acc_sus,YouTube reserves the right to suspend or termi...,-1: When the company reserves the right to sus...,0,0
3,YouTube,acc_del,YouTube reserves the right to suspend or termi...,-1: When the company reserves the right to del...,0,0
4,YouTube,ltd,"To the extent permitted by applicable law, You...",-1: Unlawful limitation of liablility for dama...,0,0
5,YouTube,ltd_cap,YouTube and its Affiliates’ total liability fo...,-1: Everyone entitled to the damages connected...,0,-1
6,YouTube,contr_chg,We may change this Agreement (1) to reflect ch...,-1: When the company reserves the right to cha...,0,1
7,YouTube,sever,If it turns out that a particular term of this...,-1: -\n0: The ToS contains provisions that kee...,0,0
8,SQUARE ENIX,serv_chg,Square Enix may immediately suspend or termina...,-1: When the company reserves the right to cha...,-1,-1
9,SQUARE ENIX,acc_sus,Square Enix may suspend or terminate your Acco...,-1: When the company reserves the right to sus...,-1,-1


Full evaluation run

In [22]:
# Full run (uncomment when ready):
full_eval_df, full_macro_f1 = run_llm_evaluation(output_filename="evaluated_tos_full.csv")
full_eval_df.head()

Starting inference for 1448 rows...
[1/1448] YouTube | serv_chg -> 0
[2/1448] YouTube | cnt_del -> 0
[3/1448] YouTube | acc_sus -> 0
[4/1448] YouTube | acc_del -> 0
[5/1448] YouTube | ltd -> 0
[6/1448] YouTube | ltd_cap -> -1
[7/1448] YouTube | contr_chg -> 1
[8/1448] YouTube | sever -> 0
[9/1448] SQUARE ENIX | serv_chg -> -1
[10/1448] SQUARE ENIX | acc_sus -> -1
[11/1448] SQUARE ENIX | acc_del -> -1
[12/1448] SQUARE ENIX | price_chg -> 0
[13/1448] SQUARE ENIX | cnt_del -> -1
[14/1448] SQUARE ENIX | arb -> -1
[15/1448] SQUARE ENIX | class -> -1
[16/1448] SQUARE ENIX | as_is -> -1
[17/1448] SQUARE ENIX | ltd -> -1
[18/1448] SQUARE ENIX | ltd_cap -> -1
[19/1448] SQUARE ENIX | contr_chg -> -1
[20/1448] SQUARE ENIX | c_law -> -1
[21/1448] SQUARE ENIX | c_forum -> -1
[22/1448] SQUARE ENIX | sever -> 0
[23/1448] Vinted | contr_chg -> 1
[24/1448] Vinted | price_chg -> 0
[25/1448] Vinted | acc_sus -> 0
[26/1448] Vinted | acc_del -> 0
[27/1448] Vinted | acc_sus -> 0
[28/1448] Vinted | acc_sus -

,Company,Classification_Code,Referenced_Text,Specific_Rubric,Ground_Truth_Score,Predicted_Score
0,YouTube,serv_chg,YouTube is constantly changing and improving t...,-1: When the company reserves the right to cha...,0,0
1,YouTube,cnt_del,If we reasonably believe that any of your Cont...,-1: When a company reserves a right to delete ...,0,0
2,YouTube,acc_sus,YouTube reserves the right to suspend or termi...,-1: When the company reserves the right to sus...,0,0
3,YouTube,acc_del,YouTube reserves the right to suspend or termi...,-1: When the company reserves the right to del...,0,0
4,YouTube,ltd,"To the extent permitted by applicable law, You...",-1: Unlawful limitation of liablility for dama...,0,0
